# MetaAlgorithmGA Testing on 1K-Node Clustered Graph

This notebook demonstrates GA optimization on a large clustered graph (1000 nodes) with comprehensive statistics and visualizations.

## Setup & Imports

In [1]:
import sys

from tests.fixtures.graphs import _create_clustered_graph

sys.path.insert(0, '..')

from meta.core import CanonicalVector, FitnessEvaluator, MetaAlgorithmGA
from src.graph.graph_manager import GraphManager
import matplotlib.pyplot as plt
import numpy as np
import time

## Helper Functions

In [2]:
def fixture_to_graph(fixture_dict) -> GraphManager:
    """Convert fixture dictionary to GraphManager."""
    graph = GraphManager.create_empty_graph()
    for v in fixture_dict['vertices']:
        graph.add_vertex(v)
    for u, v, w in fixture_dict['edges']:
        graph.add_edge(u, v, float(w))
    return graph

def format_time(seconds: float) -> str:
    """Format time in human-readable form."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    else:
        return f"{seconds/60:.1f}m"

def get_optimal_weight(fixture_dict) -> float:
    """Compute optimal matching weight using NetworkX."""
    try:
        import networkx as nx
        G = nx.Graph()
        for v in fixture_dict['vertices']:
            G.add_node(v)
        for u, v, w in fixture_dict['edges']:
            G.add_edge(u, v, weight=float(w))
        matching = nx.max_weight_matching(G, weight='weight', maxcardinality=False)
        return sum(G[u][v].get('weight', 1.0) for u, v in matching)
    except Exception:
        return 0.0

def get_algorithm_baselines(graph: GraphManager) -> dict:
    """Get baseline results from all three algorithms."""
    from src.simulation.centralized_orchestrator import CentralizedOrchestrator
    
    vector = CanonicalVector(
        luby_base_probability=0.5,
        luby_coeff_degree=0.0,
        luby_coeff_neighbors_unmatched=0.0,
        luby_coeff_clustering=0.0,
        luby_coeff_matched=0.0,
        luby_coeff_round=0.0,
        luby_coeff_weight=0.0,
        itai_timeout_rounds=5,
        max_iterations=10,
        convergence_threshold=0.05,
    )
    
    try:
        orchestrator = CentralizedOrchestrator()
        orchestrator.setup(graph)
        matching = orchestrator.run_until_convergence(max_rounds=10)
        
        if matching:
            weight = sum(graph.get_edge_weight(u, v) for u, v in matching.items() if u < v)
            return {'baseline': weight}
    except Exception:
        pass
    
    return {'baseline': 0.0}

## Define Graph Size and GA Parameters

In [3]:
POPULATION_SIZE = 20
GENERATIONS = 10
SEED = 42
NR_OF_NODES = 10000

## Test: GA Optimization on 1K-Node Clustered Graph

In [ ]:
# Load graph
print("="*80)
print(f"LOADING {NR_OF_NODES}-NODE CLUSTERED GRAPH (Seed: {SEED})")
print("="*80)


fixture = _create_clustered_graph(nr_of_nudes=NR_OF_NODES, seed=SEED)
graph = fixture_to_graph(fixture)

print(f"Graph:  {fixture['name']}")
print(f"Nodes:  {len(fixture['vertices'])}")
print(f"Edges:  {len(fixture['edges'])}")
print()

# Get optimal weight
print("Computing optimal matching weight (NetworkX)...", end=" ", flush=True)
optimal = get_optimal_weight(fixture)
print(f"✓ {optimal:.0f}")

# Get baseline
print("Getting baseline from standard algorithms...", end=" ", flush=True)
baseline_result = get_algorithm_baselines(graph)
baseline = baseline_result['baseline']
print(f"✓ {baseline:.0f}")
print()

LOADING 10000-NODE CLUSTERED GRAPH (Seed: 42)
Graph:  Clustered Graph with Communities (10000 nodes)
Nodes:  10000
Edges:  39945

Computing optimal matching weight (NetworkX)... 

## Run GA Optimization

In [ ]:
print("="*80)
print("RUNNING GA OPTIMIZATION")
print("="*80)
print()

evaluator = FitnessEvaluator()
ga = MetaAlgorithmGA(
    fitness_evaluator=evaluator,
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    mutation_rate=0.15
)

print("GA Parameters:")
print(f"  Population size: {POPULATION_SIZE}")
print(f"  Generations:     {GENERATIONS}")
print(f"  Mutation rate:   0.15")
print()

print("Running GA...", flush=True)
start = time.time()
best_vector, fitness_history = ga.evolve(graph)
elapsed = time.time() - start

best = fitness_history[-1]
gap = ((optimal - best) / (optimal + 1e-10)) * 100
improvement = ((best - baseline) / (baseline + 1e-10)) * 100

print(f"✓ Done in {format_time(elapsed)}")
print()

## Results

In [ ]:
print("="*80)
print("RESULTS SUMMARY")
print("="*80)
print()

print("Fitness Comparison:")
print(f"  Baseline (standard algorithms):  {baseline:>12.0f}")
print(f"  GA best found:                   {best:>12.0f}")
print(f"  Optimal (NetworkX):              {optimal:>12.0f}")
print()

print("Quality Metrics:")
print(f"  GA vs Baseline:                  {improvement:>12.1f}%")
print(f"  GA vs Optimal (gap):             {gap:>12.1f}%")
print()

print("Execution Time:")
print(f"  Total time:                      {elapsed:>12.1f}s")
print(f"  Time per generation:             {elapsed/10:>12.1f}s")
print()

## Fitness Progression

In [ ]:
# Plot fitness progression
fig, ax = plt.subplots(figsize=(12, 6))

gens = list(range(1, len(fitness_history) + 1))

# Plot GA fitness progression
ax.plot(gens, fitness_history, 'b-o', linewidth=2.5, markersize=10, label='GA Fitness')

# Add baseline line
ax.axhline(y=baseline, color='r', linestyle='--', linewidth=2.5, label=f'Baseline ({baseline:.0f})')

# Add optimal line
ax.axhline(y=optimal, color='g', linestyle='--', linewidth=2.5, label=f'Optimal ({optimal:.0f})')

ax.set_xlabel('Generation', fontsize=12, fontweight='bold')
ax.set_ylabel('Fitness (Matching Weight)', fontsize=12, fontweight='bold')
ax.set_title('GA Fitness Progression - 1K-Node Clustered Graph', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')

plt.tight_layout()
plt.show()

## Generation-by-Generation Analysis

In [ ]:
print("="*80)
print("GENERATION-BY-GENERATION ANALYSIS")
print("="*80)
print()

improvements = [0.0] + [fitness_history[i] - fitness_history[i-1] for i in range(1, len(fitness_history))]

print(f"{'Gen':>3} {'Fitness':>12} {'Change':>12} {'% Change':>12} {'Gap to Opt':>12}")
print("-" * 65)

for gen, fitness in enumerate(fitness_history, 1):
    change = improvements[gen-1]
    pct_change = (change / (fitness_history[0] + 1e-10)) * 100
    gap_to_opt = ((optimal - fitness) / (optimal + 1e-10)) * 100
    print(f"{gen:>3} {fitness:>12.0f} {change:>12.0f} {pct_change:>11.2f}% {gap_to_opt:>11.2f}%")

print("-" * 65)
total_improvement = fitness_history[-1] - fitness_history[0]
total_pct = (total_improvement / (fitness_history[0] + 1e-10)) * 100
print(f"Total improvement: {total_improvement:.0f} ({total_pct:+.2f}%)")
print()

## Best Parameters Found

In [ ]:
print("="*80)
print("BEST PARAMETERS FOUND BY GA")
print("="*80)
print()

bv = best_vector

print("Luby Parameters:")
print(f"  Base probability:              {bv.luby_base_probability:>12.3f}")
print(f"  Degree coefficient:            {bv.luby_coeff_degree:>12.3f}")
print(f"  Neighbors unmatched coeff:     {bv.luby_coeff_neighbors_unmatched:>12.3f}")
print(f"  Clustering coefficient:        {bv.luby_coeff_clustering:>12.3f}")
print(f"  Matched coefficient:           {bv.luby_coeff_matched:>12.3f}")
print(f"  Round coefficient:             {bv.luby_coeff_round:>12.3f}")
print(f"  Weight coefficient:            {bv.luby_coeff_weight:>12.3f}")
print()

print("Itai-Israeli Parameters:")
print(f"  Timeout rounds:                {bv.itai_timeout_rounds:>12.0f}")
print()

print("Global Parameters:")
print(f"  Max iterations:                {bv.max_iterations:>12.0f}")
print(f"  Convergence threshold:         {bv.convergence_threshold:>12.4f}")
print()

## Performance Visualization

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fitness comparison
values = [baseline, best, optimal]
labels = [f'Baseline\n({baseline:.0f})', f'GA Best\n({best:.0f})', f'Optimal\n({optimal:.0f})']
colors = ['red', 'blue', 'green']

axes[0].bar(labels, values, color=colors, alpha=0.7, width=0.6)
axes[0].set_ylabel('Matching Weight', fontweight='bold', fontsize=11)
axes[0].set_title('Fitness Comparison', fontweight='bold', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values):
    axes[0].text(i, v + 100, f"{v:.0f}", ha='center', fontweight='bold', fontsize=10)

# Plot 2: Quality metrics
metrics = ['GA vs Baseline', 'Gap to Optimal']
values_metrics = [improvement, gap]
colors_metrics = ['green' if v > 0 else 'red' for v in values_metrics]

bars = axes[1].bar(metrics, values_metrics, color=colors_metrics, alpha=0.7, width=0.6)
axes[1].set_ylabel('Percentage (%)', fontweight='bold', fontsize=11)
axes[1].set_title('Quality Metrics', fontweight='bold', fontsize=12)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(values_metrics):
    offset = 0.5 if v > 0 else -0.5
    axes[1].text(i, v + offset, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## Statistics Summary

In [ ]:
print("="*80)
print("STATISTICS SUMMARY")
print("="*80)
print()

print("Convergence Analysis:")
# Find convergence point (when improvement < 0.1% of first generation)
convergence_gen = None
convergence_threshold = 0.001 * fitness_history[0]
for i, imp in enumerate(improvements[1:], 1):
    if abs(imp) < convergence_threshold:
        convergence_gen = i
        break

if convergence_gen:
    print(f"  Converged at generation:       {convergence_gen}")
    print(f"  Generations to convergence:    {convergence_gen}")
else:
    print(f"  Did not converge within 10 generations")

print()
print("Generation Statistics:")
print(f"  First generation fitness:      {fitness_history[0]:.0f}")
print(f"  Last generation fitness:       {fitness_history[-1]:.0f}")
print(f"  Fitness range:                 {fitness_history[-1] - fitness_history[0]:.0f}")
print(f"  Average per-generation change: {np.mean(improvements[1:]):.0f}")
print(f"  Max improvement in one gen:    {np.max(improvements):.0f}")
print()

print("Overall Performance:")
print(f"  GA improvement vs Baseline:    {improvement:>6.2f}%")
print(f"  Gap to Optimal:                {gap:>6.2f}%")
print(f"  Execution time:                {format_time(elapsed)}")
print(f"  Evaluations per second:        {(10*10) / elapsed:.1f}")
print()